# Teste de corregistro MNI → imagens ADNI (global rígido → afim + local SyN)

Móvel = MNI152 2009c; fixas = `N_POR_DIAG` imagens de cada diagnóstico em `DIAGS` (sorteio com `SEED`: 1 imagem por paciente, paciente não se repete entre diagnósticos; só imagens do `adnimerged.csv` com T1 e `regions` em disco).

Variantes (escolha em `RODAR`, célula 1):

- `sem_mask`: SyN padrão do ANTsPy, sem máscara fixa (= o que o `2.2` executou, já que `fixed_mask` é ignorado).
- `com_mask`: igual, mas com `mask` = máscara da imagem fixa.
- `syn_robusto`: sem máscara, SyN com os parâmetros do `3.1 --src oasis` (CC raio 4, `100x70x50x20`).

Saídas em `test_dvf/{DIAG}_{ID}/`, todas no grid da imagem fixa: `fixed`, `regions` (labels originais, sem transformação), `mni_warped_{variante}` e `warp_{variante}_1Warp`. No ITK-SNAP: main = `mni_warped_{variante}`, segmentação = `regions`. Registro retomável: variantes já salvas são puladas.

In [1]:
import os
import shutil

import ants
import numpy as np
import pandas as pd

INPUT_DIR = "/mnt/databases/mri/adni/preproc/4-mni-hist-matching"
REGIONS_DIR = "/mnt/databases/mri/adni/preproc/5-parcellation/regions"
MNI_PATH = "/mnt/study-data/pgirardi/datasets_img/atlases/templates/mni152_2009c_template.nii.gz"
SUFFIX = "_stripped_nlm_denoised_biascorrected_mni_template.nii.gz"
OUT_DIR = "test_dvf"
SEED = 42

DIAGS = ["CN","MCI","AD"]  # ex.: ["AD", "CN", "MCI"]
N_POR_DIAG = 30  # imagens por diagnóstico (1 por paciente)

# robustez vem da métrica/iterações, não do tipo: = _register do 3.1 --src oasis (SyNRA)
SYN_ROBUSTO = {"syn_metric": "CC", "syn_sampling": 4, "reg_iterations": (100, 70, 50, 20)}
# variante → (usar máscara da fixa, parâmetros do SyN); None = padrão ANTsPy (Mattes, (40, 20, 0))
VARIANTES = {
    "sem_mask": (False, None),
    "com_mask": (True, None),
    "syn_robusto": (False, SYN_ROBUSTO),
}
RODAR = ["sem_mask", "com_mask", "syn_robusto"]

# ponytail: só fixa seed; determinismo total exige 1 thread (set_ants_deterministic(True)), SyN fica muito lento
ants.config.set_ants_deterministic(False, SEED)
moving = ants.image_read(MNI_PATH)
moving_mask = ants.get_mask(moving)


def t1_path(img_id):
    return os.path.join(INPUT_DIR, img_id + SUFFIX)


def regions_path(img_id):
    return os.path.join(REGIONS_DIR, img_id + "_regions.nii.gz")


df = pd.read_csv("csvs/adnimerged.csv")
df = df[df["DIAG"].isin(DIAGS)]
df = df[df["ID_IMG"].map(lambda i: os.path.isfile(t1_path(i)) and os.path.isfile(regions_path(i)))]
# 1 imagem sorteada por paciente; paciente não se repete entre diagnósticos (ex.: CN que converteu para MCI)
df = df.sample(frac=1, random_state=SEED).drop_duplicates("ID_PT")
sel = pd.concat([df[df["DIAG"] == d].head(N_POR_DIAG) for d in DIAGS])
for d in DIAGS:
    n = (sel["DIAG"] == d).sum()
    if n < N_POR_DIAG:
        print(f"[WARN] {d}: só {n} imagens disponíveis")
print(sel[["ID_IMG", "ID_PT", "DIAG", "SEX", "AGE", "VISIT"]].to_string(index=False))


def pasta(row):
    return os.path.join(OUT_DIR, f"{row.DIAG}_{row.ID_IMG}")


def registrar(fixed, usar_mask, syn_kw):
    """Rigid -> Affine -> SyN (ordem do prealign_to_target do 2.2). Retorna o resultado do SyN."""
    mk = {"mask": ants.get_mask(fixed)} if usar_mask else {}
    rig = ants.registration(fixed, moving, "Rigid", moving_mask=moving_mask, **mk)
    w = rig["warpedmovout"]
    aff = ants.registration(fixed, w, "Affine", moving_mask=ants.get_mask(w), **mk)
    w = aff["warpedmovout"]
    # SyNOnly: só o deformável ("SyN" refaria o afim sobre a imagem já alinhada)
    return ants.registration(fixed, w, "SyNOnly", moving_mask=ants.get_mask(w), **mk, **(syn_kw or {}))

 ID_IMG      ID_PT DIAG SEX  AGE VISIT
 I34007 031_S_0618   CN   M   76   m06
 I23492 141_S_0767   CN   F   73    sc
I294432 041_S_4427   CN   M   72   v04
I123539 067_S_0257   CN   F   81   m24
 I37848 116_S_1232   CN   F   72    sc
I322057 129_S_4369   CN   M   69   v05
I502018 002_S_5178   CN   M   71   v21
I125201 098_S_0896   CN   M   79   m24
I324988 023_S_4448   CN   F   64   v05
 I80850 003_S_0981   CN   F   85   m12
I241008 024_S_4084   CN   F   68   v02
I225667 116_S_4010   CN   F   71   v02
I287487 002_S_1280   CN   F   76   v06
 I95117 136_S_0186   CN   F   83   m24
I436264 128_S_1242   CN   F   72   m12
 I40549 023_S_1306   CN   F   75    sc
I436705 130_S_0886   CN   F   75   m36
 I86729 023_S_1190   CN   F   78   m12
I100034 941_S_1194   CN   M   86   m12
I193526 011_S_0021   CN   F   75   m24
I229148 031_S_4021   CN   M   67   v02
I403713 011_S_0023   CN   M   80   v21
 I77837 033_S_0923   CN   M   84   m12
I243002 099_S_4104   CN   M   72   v02
I153191 033_S_0741   CN  

## 1. Registro

In [ ]:
for k, row in enumerate(sel.itertuples(), 1):
    d = pasta(row)
    os.makedirs(d, exist_ok=True)
    fixed = ants.image_read(t1_path(row.ID_IMG))
    if not os.path.isfile(os.path.join(d, "fixed.nii.gz")):
        ants.image_write(fixed, os.path.join(d, "fixed.nii.gz"))
        shutil.copy2(regions_path(row.ID_IMG), os.path.join(d, "regions.nii.gz"))
    for v in RODAR:
        out = os.path.join(d, f"mni_warped_{v}.nii.gz")
        if os.path.isfile(out):
            print(f"[{k}/{len(sel)}] [SKIP] {d} {v}")
            continue
        syn = registrar(fixed, *VARIANTES[v])
        shutil.copy2(syn["fwdtransforms"][0], os.path.join(d, f"warp_{v}_1Warp.nii.gz"))
        ants.image_write(syn["warpedmovout"], out)  # por último: marca de concluído
        print(f"[{k}/{len(sel)}] [OK] {d} {v}", flush=True)

[1/90] [SKIP] test_dvf/CN_I34007 sem_mask
[1/90] [SKIP] test_dvf/CN_I34007 com_mask
[1/90] [SKIP] test_dvf/CN_I34007 syn_robusto
[2/90] [SKIP] test_dvf/CN_I23492 sem_mask
[2/90] [SKIP] test_dvf/CN_I23492 com_mask
[2/90] [SKIP] test_dvf/CN_I23492 syn_robusto
[3/90] [SKIP] test_dvf/CN_I294432 sem_mask
[3/90] [SKIP] test_dvf/CN_I294432 com_mask
[3/90] [SKIP] test_dvf/CN_I294432 syn_robusto
[4/90] [SKIP] test_dvf/CN_I123539 sem_mask
[4/90] [SKIP] test_dvf/CN_I123539 com_mask
[4/90] [SKIP] test_dvf/CN_I123539 syn_robusto
[5/90] [SKIP] test_dvf/CN_I37848 sem_mask
[5/90] [SKIP] test_dvf/CN_I37848 com_mask
[5/90] [SKIP] test_dvf/CN_I37848 syn_robusto
[6/90] [SKIP] test_dvf/CN_I322057 sem_mask
[6/90] [SKIP] test_dvf/CN_I322057 com_mask
[6/90] [SKIP] test_dvf/CN_I322057 syn_robusto
[7/90] [SKIP] test_dvf/CN_I502018 sem_mask
[7/90] [SKIP] test_dvf/CN_I502018 com_mask


## 2. Métricas

In [ ]:
def resumo(d, v):
    fixed = ants.image_read(os.path.join(d, "fixed.nii.gz"))
    regions = ants.image_read(os.path.join(d, "regions.nii.gz"))
    w = ants.image_read(os.path.join(d, f"mni_warped_{v}.nii.gz"))
    jac = ants.create_jacobian_determinant_image(fixed, os.path.join(d, f"warp_{v}_1Warp.nii.gz")).numpy()
    m = ants.get_mask(fixed).numpy() > 0
    # hipocampo L/R (17/53): núcleo p/ Jacobiano (d2 inclui corno temporal, que na AD puxa J < 1);
    # dilatado 2 voxels (≈ hippocampus_d2 do 3.2) p/ correlação local
    hippo = ants.threshold_image(regions, 17, 17) + ants.threshold_image(regions, 53, 53)
    core, h = hippo.numpy() > 0, ants.iMath(hippo, "MD", 2).numpy() > 0
    wm = ants.get_mask(w).numpy() > 0
    f, x = fixed.numpy(), w.numpy()
    return {
        "ncc_cerebro": np.corrcoef(f[m], x[m])[0, 1],
        "ncc_hipocampo": np.corrcoef(f[h], x[h])[0, 1],
        "dice_cerebro": 2 * (wm & m).sum() / (wm.sum() + m.sum()),
        "jac_hipocampo_mean": jac[core].mean(),
        "jac_min": jac[m].min(),
        "jac_max": jac[m].max(),
        "jac_neg_%": 100 * (jac[m] <= 0).mean(),
        "jac_std": jac[m].std(),
    }


tab = pd.DataFrame([
    {"ID_IMG": r.ID_IMG, "DIAG": r.DIAG, "variante": v, **resumo(pasta(r), v)}
    for r in sel.itertuples() for v in RODAR
    if os.path.isfile(os.path.join(pasta(r), f"mni_warped_{v}.nii.gz"))
])
tab.to_csv(os.path.join(OUT_DIR, "resumo.csv"), index=False)
print(tab.round(4).to_string(index=False))
print("\nMédia por diagnóstico × variante:")
print(tab.groupby(["DIAG", "variante"]).mean(numeric_only=True).round(4).to_string())

ID_IMG DIAG    variante  ncc_cerebro  ncc_hipocampo  dice_cerebro  jac_hipocampo_mean  jac_min  jac_max  jac_neg_%  jac_std
 I7071   CN    sem_mask       0.7957         0.8200        0.9903              1.0181   0.3023   2.6603        0.0   0.2245
 I7071   CN syn_robusto       0.8660         0.9023        0.9810              1.1261   0.2196   3.9694        0.0   0.2813
 I7211   CN    sem_mask       0.7827         0.8637        0.9862              0.9122   0.3406   2.3244        0.0   0.2391
 I7211   CN syn_robusto       0.8600         0.9251        0.9800              1.3726   0.2357   5.1463        0.0   0.3968
 I7302   CN    sem_mask       0.7937         0.8583        0.9900              0.7917   0.4404   2.3408        0.0   0.2107
 I7302   CN syn_robusto       0.8809         0.9258        0.9792              0.8559   0.1614   6.0514        0.0   0.3122
 I7055   AD    sem_mask       0.7268         0.4841        0.9905              0.7164   0.1108   3.5665        0.0   0.3186
 I7055  

## 3. Média ± DP por diagnóstico e abordagem

1. Média ± DP (amostral) de cada métrica por diagnóstico × abordagem.
2. Diferença pareada contra `sem_mask` (mesmas imagens): média ± DP, % de imagens com Δ > 0 e p do Wilcoxon. "% Δ > 0" só indica o sentido: correlação e Dice, maior é melhor; Jacobiano não tem "melhor".
3. Kruskal–Wallis entre diagnósticos, por abordagem: o Jacobiano e a correlação no hipocampo separam CN, MCI e AD?

São muitos valores de p (2 abordagens × 3 diagnósticos × 8 métricas): tratar como exploratórios ou corrigir (Holm). Salva `test_dvf/resumo_media_dp.csv` e `test_dvf/resumo_pareado.csv`.

In [ ]:
from scipy.stats import kruskal, wilcoxon

METRICAS = ["ncc_cerebro", "ncc_hipocampo", "dice_cerebro", "jac_hipocampo_mean",
            "jac_min", "jac_max", "jac_neg_%", "jac_std"]
ORDEM_DIAG = [d for d in ("CN", "MCI", "AD") if d in set(tab["DIAG"])]
ORDEM_VAR = [v for v in ("sem_mask", "com_mask", "syn_robusto") if v in set(tab["variante"])]
BASE = "sem_mask"


def fmt3(df):
    return df.apply(lambda s: s.map("{:.3f}".format))


# 1) média ± DP por diagnóstico × abordagem (DP amostral, ddof=1)
g = tab.groupby(["DIAG", "variante"])[METRICAS]
media_dp = fmt3(g.mean()) + " ± " + fmt3(g.std())
media_dp.insert(0, "n", g.size())
media_dp = media_dp.reindex(pd.MultiIndex.from_product([ORDEM_DIAG, ORDEM_VAR], names=["DIAG", "abordagem"]))
media_dp.to_csv(os.path.join(OUT_DIR, "resumo_media_dp.csv"))
print("Média ± DP por diagnóstico e abordagem")
display(media_dp)

# 2) diferença pareada (mesmas imagens) contra sem_mask
w = tab.pivot_table(index=["DIAG", "ID_IMG"], columns="variante", values=METRICAS)
linhas = []
for v in (x for x in ORDEM_VAR if x != BASE):
    for d in ORDEM_DIAG:
        for c in METRICAS:
            par = w.loc[d, [(c, BASE), (c, v)]].dropna()
            delta = par[(c, v)] - par[(c, BASE)]
            p = wilcoxon(delta).pvalue if len(delta) and (delta != 0).any() else 1.0
            linhas.append({
                "DIAG": d, "abordagem": f"{v} − {BASE}", "metrica": c, "n": len(delta),
                "delta_media": delta.mean(), "delta_dp": delta.std(),
                "pct_delta_pos": 100 * (delta > 0).mean(), "p_wilcoxon": p,
            })
pareado = pd.DataFrame(linhas)
pareado.to_csv(os.path.join(OUT_DIR, "resumo_pareado.csv"), index=False)
pareado["txt"] = (pareado["delta_media"].map("{:+.3f}".format) + " ± " + pareado["delta_dp"].map("{:.3f}".format)
                  + " (" + pareado["pct_delta_pos"].map("{:.0f}".format) + "% >0; p="
                  + pareado["p_wilcoxon"].map("{:.1e}".format) + ")")
print(f"\nDiferença pareada contra {BASE}: média ± DP (% imagens com Δ > 0; p Wilcoxon)")
display(pareado.pivot_table(index=["abordagem", "DIAG"], columns="metrica", values="txt", aggfunc="first")
        .reindex(index=pd.MultiIndex.from_product([pareado["abordagem"].unique(), ORDEM_DIAG]), columns=METRICAS))

# 3) diagnósticos se separam? (Kruskal–Wallis por abordagem)
print("\nKruskal–Wallis entre diagnósticos (p):")
for v in ORDEM_VAR:
    sub = tab[tab["variante"] == v]
    ps = {c: kruskal(*[sub.loc[sub["DIAG"] == d, c] for d in ORDEM_DIAG]).pvalue
          for c in ("jac_hipocampo_mean", "ncc_hipocampo")}
    print(f"  {v:12s} " + "  ".join(f"{c}: {p:.1e}" for c, p in ps.items()))

## Justificativa e resultados do teste de máscara (1 imagem)

**Objetivo.** Verificar a afirmação da auditoria de que `fixed_mask`, usado em `2.2_groupwise_ants.py`, é ignorado pelo ANTsPy, e medir quanto a máscara da imagem fixa (`mask`) altera o corregistro.

**Evidência no código (antspyx 0.6.3).**

- A assinatura de `ants.registration` tem `mask` (fixa), `moving_mask` e `mask_all_stages`, mas não tem `fixed_mask`. O argumento cai em `**kwargs`, que não é usado: é descartado sem erro nem aviso.
- As máscaras viram a opção `-x [fixa,móvel]` do `antsRegistration`; sem `mask`, a parte fixa é `NA`. Em transformações com vários estágios (`SyN`, `SyNRA`), sem `mask_all_stages=True` a máscara só vale no último estágio.
- Portanto, a variante `sem_mask` reproduz exatamente o que o 2.2 executou: só `moving_mask`.

**Desenho.** MNI152 2009c (móvel) → I7055 (fixa; AD, 011_S_0003, 81 anos). Rigid → Affine → SyNOnly, na mesma ordem de `prealign_to_target` do 2.2. A única diferença entre `sem_mask` e `com_mask` é a máscara da imagem fixa (`ants.get_mask`).

**Resultados (I7055).** A configuração sem máscara foi executada duas vezes (teste original e rodada multi-imagem), o que dá uma medida da variação entre execuções.

| Métrica | sem `mask` (1ª exec.) | sem `mask` (2ª exec.) | com `mask` |
|---|---|---|---|
| Correlação de intensidade no cérebro | 0,709 | 0,727 | 0,694 |
| Dice do cérebro | 0,990 | 0,991 | 0,970 |
| Jacobiano mín. / máx. | 0,128 / 4,03 | 0,111 / 3,57 | 0,121 / 4,26 |
| Voxels com Jacobiano ≤ 0 | 0% | 0% | 0% |
| DP do Jacobiano | 0,330 | 0,319 | 0,318 |

Diferença entre sem e com `mask` (1ª execução): parâmetros do rígido até 1,87 e do afim até 3,61 (provavelmente translação, em mm); deslocamento do SyN com diferença média de 1,85 mm (p95 3,33 mm; máx. 11,8 mm).

**Interpretação.**

- Repetir a mesma configuração sem máscara mudou a correlação em 0,018 e o Jacobiano máximo em 0,46: é a variação entre execuções (ANTs com várias threads não é determinístico). A queda de correlação com máscara (0,015) e as diferenças de Jacobiano estão dentro dessa variação.
- O único efeito claro da máscara é a queda do Dice do cérebro (0,020, contra 0,0005 entre execuções), e é para pior. Explicação provável: as imagens já estão sem crânio, com fundo zero; sem máscara, a borda cérebro/fundo entra na métrica e ajuda a alinhar o contorno, e a máscara corta essa borda.
- A máscara muda o registro já nos estágios lineares; parte da diferença no deslocamento do SyN é compensação dos alinhamentos lineares diferentes, não diferença anatômica.
- A configuração sem máscara (a que o 2.2 executou) gerou deformações válidas nas 6 imagens da rodada multi-imagem (3 CN, 3 AD): nenhum Jacobiano ≤ 0, Dice do cérebro 0,97–0,99, correlação 0,73–0,80.

**Conclusões.**

1. O bug é real, mas só afeta o `2.2_groupwise_ants.py` (templates ADNI, usados nas referências `disp_ad` e `disp_cnad`). Esses templates foram construídos, na prática, sem máscara fixa, e o teste indica que isso não é prejudicial. Correção recomendada: remover `fixed_mask` e documentar "sem máscara fixa", em vez de trocar por `mask`.
2. O pipeline OASIS não passa máscara em nenhum registro: o rígido do `2.3` usa `corregistro_rigid_mni` sem máscara, o `ants.build_template` recebe só `TEMPLATE_REG`, e o `SyNRA` do `3.1 --src oasis` é chamado sem `mask`/`moving_mask` (equivale a `-x [NA,NA]`). O que foi executado é o que estava especificado; o bug não se aplica.
3. Limitações: `com_mask` só foi executado em 1 imagem; configuração diferente da do OASIS (MNI como móvel, métrica Mattes, `SyNOnly` 40x20x0); variação entre execuções estimada com uma única repetição. Para generalizar, rodar `RODAR = ["sem_mask", "com_mask"]` com mais imagens (`DIAGS`, `N_POR_DIAG`).

## Justificativa e resultados do SyN robusto (parâmetros do `3.1 --src oasis`), 3 CN e 3 AD

Mesmo rígido → afim, sem máscara fixa. Muda só o estágio deformável:

- `sem_mask` (padrão): Mattes, `reg_iterations=(40, 20, 0)`, sem iterações na resolução máxima.
- `syn_robusto`: CC raio 4, `reg_iterations=(100, 70, 50, 20)`, igual ao `SyNRA` do pipeline OASIS. O tipo continua `SyNOnly`: no ANTsPy o tipo só escolhe os estágios lineares; a robustez vem da métrica e das iterações.

Critério de melhora: `ncc_hipocampo` e `ncc_cerebro` maiores, sem Jacobiano ≤ 0. `jac_hipocampo_mean` é calculado no núcleo do hipocampo (sem dilatação); J < 1 = estrutura maior na imagem fixa que no MNI.

**Sujeitos.** CN: I7071 (M, 74), I7211 (F, 85), I7302 (M, 66). AD: I7055 (M, 81), I7161 (M, 75), I8023 (M, 64). Primeira imagem de cada paciente na ordem do `adnimerged.csv` (visita `sc`).

**Resultados** (média [mín–máx] por grupo; `test_dvf/resumo.csv`).

| Métrica | CN padrão | CN robusto | AD padrão | AD robusto |
|---|---|---|---|---|
| Correlação no cérebro | 0,791 [0,783–0,796] | 0,869 [0,860–0,881] | 0,756 [0,727–0,803] | 0,843 [0,807–0,874] |
| Correlação no hipocampo (d2) | 0,847 [0,820–0,864] | 0,918 [0,902–0,926] | 0,651 [0,484–0,777] | 0,767 [0,635–0,842] |
| Dice do cérebro | 0,989 | 0,980 | 0,978 | 0,964 |
| J no hipocampo (núcleo) | 0,91 [0,79–1,02] | 1,12 [0,86–1,37] | 0,77 [0,72–0,88] | 0,81 [0,73–0,93] |
| J mín. / máx. | 0,36 / 2,44 | 0,21 / 5,06 | 0,24 / 3,20 | 0,15 / 4,39 |
| Voxels com J ≤ 0 | 0% | 0% | 0% | 0% |

**Interpretação.**

- **Correspondência melhora em todas as imagens.** A correlação no cérebro subiu nas 6 (+0,07 a +0,11) e no hipocampo também nas 6 (+0,06 a +0,15). O ganho no hipocampo foi maior na AD (+0,116 contra +0,070 no CN), justamente onde o padrão era pior (I7055: 0,48 → 0,63). Esses ganhos são 3 a 8 vezes a variação entre execuções medida no teste de máscara (~0,02).
- **A AD continua pior no hipocampo** (0,77 contra 0,92 no CN): com atrofia acentuada, a correspondência local com um template de adultos jovens segue imperfeita mesmo com o SyN robusto.
- **Dice do cérebro cai levemente** (−0,009 no CN, −0,014 na AD): a CC prioriza a estrutura interna em vez do contorno. Fica ≥ 0,95 em todas.
- **Deformações mais intensas, mas válidas:** a faixa do Jacobiano alarga (máx. até 6,1) e nenhum voxel tem J ≤ 0.
- **Sinal do Jacobiano confirmado:** nos ventrículos laterais J < 1 em todas as imagens (0,32–0,81), e menor na AD com ventrículos maiores (I7055, 77 cm³: 0,32–0,36).
- **J no hipocampo não seguiu a expectativa:** subiu mais no CN (+0,21) que na AD (+0,04), e na AD ficou abaixo de 1 nas duas variantes, próximo do J do corno temporal vizinho (0,60–0,91). Nem o SyN robusto separa a atrofia do hipocampo do aumento do corno temporal adjacente quando a referência é o MNI de adultos jovens.

**Conclusões.**

1. Em imagens atróficas, o SyN robusto (CC raio 4, `100x70x50x20`) registra melhor que o padrão do ANTsPy: melhora a correspondência global e hipocampal em todas as imagens, mais na AD, sem dobras. Isso sustenta a configuração do pipeline OASIS e indica que os registros com SyN padrão (`2.2` e o caminho ADNI do `3.1`, `type_of_transform="SyN"` sem parâmetros) têm correspondência local pior.
2. O Jacobiano médio no hipocampo contra o MNI não é um marcador direto de atrofia. Deve ser lido de forma relativa (por exemplo `logjac_rel_mean` do `3.2`, que desconta a deformação global) e contra templates normativos pareados por idade, como os templates OASIS por década.
3. Limitações: 3 imagens por grupo; seleção pela ordem do CSV, não aleatória; MNI como móvel, e não o template OASIS; execução com várias threads.

**Tempo observado:** ~15 min por imagem no robusto contra ~1 min no padrão (várias threads, servidor compartilhado).